# Arizona Diamondbacks Radio Affiliates
### SBE Chapter 9 Demo — What Can You Do With Python + Broadcast Data?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/W2NJL/sbe9-demo/blob/main/SBE_Dbacks_Radio_Affiliates_Demo.ipynb)

In this notebook we'll explore a CSV exported from the **Radioland** broadcast database:
- `dbacks_affiliates.csv` — every AM & FM station carrying Arizona Diamondbacks games

We'll walk through **8 steps** — loading data, counting stations, comparing AM vs FM,
mapping affiliate locations, measuring geographic reach, and more.

> **How to run:** Click each code cell and press **Shift + Enter** (or the Play button) to execute it.

---
## Step 1 — Load the CSV File From GitHub

Pandas can read a CSV straight from a URL — no downloads needed.
We point it at the raw GitHub link and we're off.

In [ ]:
import pandas as pd

DBACKS_URL = 'https://raw.githubusercontent.com/W2NJL/sbe9-demo/main/dbacks_affiliates.csv'

dbacks_raw = pd.read_csv(DBACKS_URL)

print(f'Rows loaded: {len(dbacks_raw)}')
print()
print('Columns:', list(dbacks_raw.columns))
dbacks_raw.head()

---
## Step 2 — Clean the Data (Deduplication)

In the RadioLand database, AM stations often have **multiple rows** — one for their daytime pattern and one
for nighttime (different power levels, sometimes different antenna arrays).
We'll keep one row per station, using the **highest power** value so our
counts and charts reflect unique stations.

This is a very common first step in data analysis — real-world data is
almost never perfectly clean out of the box.

In [ ]:
def dedup(df):
    """Keep one row per station (callsign + band), choosing the highest power."""
    df = df.copy()
    df['erp'] = pd.to_numeric(df['erp'], errors='coerce')
    df = df.sort_values('erp', ascending=False)
    df = df.drop_duplicates(subset=['callsign', 'band'], keep='first')
    return df.sort_values(['band', 'callsign']).reset_index(drop=True)

dbacks = dedup(dbacks_raw)

print(f'Unique stations: {len(dbacks)}  (was {len(dbacks_raw)} rows before dedup)')
print()
dbacks[['callsign', 'frequency', 'band', 'city', 'state', 'erp', 'format']].head(15)

---
## Step 3 — AM vs FM Breakdown

How does the network split between AM and FM?
`.value_counts()` is one of the most useful Pandas methods.

In [ ]:
import matplotlib.pyplot as plt

n_am = len(dbacks[dbacks['band'] == 'AM'])
n_fm = len(dbacks[dbacks['band'] == 'FM'])
total = len(dbacks)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = axes[0].bar(['AM', 'FM'], [n_am, n_fm],
                   color=['#A71930', '#E3D4AD'], edgecolor='black', width=0.4)
axes[0].set_ylabel('Number of Stations')
axes[0].set_title('D-backs Affiliates: AM vs FM')
for bar, count in zip(bars, [n_am, n_fm]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 str(count), ha='center', fontweight='bold', fontsize=14)

# Pie chart
axes[1].pie([n_am, n_fm], labels=['AM', 'FM'], autopct='%1.0f%%',
            colors=['#A71930', '#E3D4AD'], startangle=90,
            textprops={'fontsize': 13})
axes[1].set_title(f'D-backs Radio Network ({total} stations total)')

plt.tight_layout()
plt.show()

print(f'AM stations: {n_am}  ({n_am/total*100:.0f}%)')
print(f'FM stations: {n_fm}  ({n_fm/total*100:.0f}%)')
print(f'Total:       {total}')

---
## Step 4 — Which States Are Covered?

Which states have the highest concentration of Diamondbacks affiliates?
The D-backs broadcast footprint tends to be concentrated in the Southwest.

In [ ]:
state_counts = dbacks['state'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
state_counts.plot(kind='barh', ax=ax, color='#A71930', edgecolor='black')
ax.set_title('D-backs Affiliates by State')
ax.set_xlabel('Number of Stations')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f'D-backs affiliates span {dbacks["state"].nunique()} states/territories:')
print(', '.join(state_counts.index))

---
## Step 5 — Format Breakdown

What formats do the affiliate stations carry? Sports radio is the obvious guess —
but AM affiliates in smaller markets sometimes wear multiple hats.

In [ ]:
format_counts = dbacks['format'].fillna('Unknown').value_counts().head(12)

fig, ax = plt.subplots(figsize=(10, 5))
format_counts.plot(kind='barh', ax=ax, color='#E3D4AD', edgecolor='#A71930')
ax.set_title('D-backs Affiliates by Station Format')
ax.set_xlabel('Number of Stations')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(format_counts.to_string())

---
## Step 6 — Interactive Map With Folium

**Folium** renders [Leaflet.js](https://leafletjs.com/) maps right inside the
notebook. Each station gets a color-coded marker — click one to see its
callsign, frequency, city, and power.

- **Red** markers = AM stations
- **Sand/gold** markers = FM stations

Folium comes **pre-installed** on Google Colab, so no `pip install` needed.

In [ ]:
import folium

# Center on Arizona
m = folium.Map(location=[34.5, -112.0], zoom_start=6, tiles='CartoDB positron')

band_colors = {'AM': '#A71930', 'FM': '#C0A060'}

for _, row in dbacks.iterrows():
    if pd.isna(row['lat']) or pd.isna(row['lon']):
        continue
    lon = -abs(row['lon'])
    color = band_colors.get(row['band'], '#888888')
    popup_html = (
        f"<b>{row['callsign']}</b><br>"
        f"{row['frequency']} {row['band']}<br>"
        f"{row.get('city', '')} {row.get('state', '')}<br>"
        f"ERP: {row['erp']} {'kW' if row['band']=='FM' else 'W'}<br>"
        f"Format: {row.get('format', 'N/A')}"
    )
    folium.CircleMarker(
        location=[row['lat'], lon],
        radius=7,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.75,
        popup=folium.Popup(popup_html, max_width=240),
        tooltip=f"{row['callsign']} {row['frequency']} ({row['band']})"
    ).add_to(m)

legend_html = '''
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
            background:white; padding:12px 16px; border:2px solid #A71930;
            border-radius:6px; font-size:14px; font-family:sans-serif;">
  <b style="color:#A71930;">D-backs Radio Network</b><br><br>
  <i style="color:#A71930;">&#11044;</i> AM stations<br>
  <i style="color:#C0A060;">&#11044;</i> FM stations
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

m

---
## Step 7 — Geographic Reach: How Far Do Affiliates Spread?

We measure the distance from each affiliate to Chase Field (Phoenix) using the
**Haversine formula** — the standard way to calculate great-circle distance on a sphere.

This shows the true geographic footprint of the D-backs broadcast network.

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance in miles between two lat/lon points."""
    R = 3958.8
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

# Chase Field, Phoenix AZ
CHASE_FIELD = (33.4453, -112.0667)

dbacks_d = dbacks.copy()
dbacks_d['distance_mi'] = dbacks_d.apply(
    lambda r: haversine(r['lat'], -abs(r['lon']), CHASE_FIELD[0], CHASE_FIELD[1])
              if pd.notna(r['lat']) and pd.notna(r['lon']) else None,
    axis=1
)

dist = dbacks_d['distance_mi'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram by band
am_dist = dbacks_d[dbacks_d['band']=='AM']['distance_mi'].dropna()
fm_dist = dbacks_d[dbacks_d['band']=='FM']['distance_mi'].dropna()
axes[0].hist(am_dist, bins=15, alpha=0.7, label='AM', color='#A71930', edgecolor='black')
axes[0].hist(fm_dist, bins=15, alpha=0.7, label='FM', color='#E3D4AD', edgecolor='black')
axes[0].set_xlabel('Distance from Chase Field (miles)')
axes[0].set_ylabel('Number of Stations')
axes[0].set_title('D-backs Affiliates: Distance from Chase Field')
axes[0].legend()

# Top 10 farthest
farthest = dbacks_d.nlargest(10, 'distance_mi')[['callsign', 'frequency', 'band', 'city', 'state', 'distance_mi']]
farthest['distance_mi'] = farthest['distance_mi'].round(0).astype(int)
farthest.plot(kind='barh', x='callsign', y='distance_mi', ax=axes[1],
              color='#A71930', edgecolor='black', legend=False)
axes[1].set_xlabel('Distance (miles)')
axes[1].set_title('Top 10 Farthest D-backs Affiliates from Chase Field')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f'Total affiliates with coordinates: {len(dist)}')
print(f'Average distance from Chase Field: {dist.mean():.0f} miles')
print(f'Farthest affiliate:                {dist.max():.0f} miles')
print(f'Closest affiliate:                 {dist.min():.0f} miles')
print()

far_row = dbacks_d.loc[dbacks_d['distance_mi'].idxmax()]
print(f'Farthest: {far_row["callsign"]} {far_row["frequency"]} {far_row["band"]}'
      f' — {far_row["city"]}, {far_row["state"]} ({far_row["distance_mi"]:.0f} mi)')

---
## Step 8 — Summary Scorecard

Let's pull everything together into a single summary table.

In [ ]:
am = dbacks[dbacks['band'] == 'AM']
fm = dbacks[dbacks['band'] == 'FM']

scorecard = pd.DataFrame({
    'Metric': [
        'Total Affiliates',
        'AM Stations',
        'FM Stations',
        'AM Percentage',
        'States / Territories Covered',
        'Avg Distance from Chase Field (mi)',
        'Farthest Affiliate (mi)',
        'Highest-Power AM (W)',
        'Avg FM ERP (kW)',
    ],
    'Arizona Diamondbacks': [
        len(dbacks),
        n_am,
        n_fm,
        f'{n_am / total * 100:.0f}%',
        dbacks['state'].nunique(),
        f'{dist.mean():.0f}',
        f'{dist.max():.0f}',
        f"{am['erp'].max():.0f}" if len(am) else 'N/A',
        f"{fm['erp'].mean():.1f}" if len(fm) else 'N/A',
    ],
})

scorecard_styled = scorecard.style.set_properties(**{
    'text-align': 'center',
    'font-size': '14px',
    'padding': '6px 12px',
}).set_properties(
    subset=['Metric'], **{'text-align': 'left', 'font-weight': 'bold'}
).hide(axis='index')

display(scorecard_styled)

print()
print(f'The D-backs radio network covers {dbacks["state"].nunique()} states with {total} stations.')
print(f'AM makes up {n_am/total*100:.0f}% of the network — typical for a Southwest baseball club.')
print(f'States covered: {', '.join(sorted(dbacks["state"].dropna().unique()))}')